In [9]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, average_precision_score

In [3]:
# 전처리 마친 데이터 불러오기
train_df = pd.read_csv("train_e.csv")
test_df = pd.read_csv("test_e.csv")
X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

In [7]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
xgb_model = XGBClassifier(
    # colsample_bytree=0.8,
    # learning_rate=0.03,
    # max_depth=3,
    # min_child_weight=8,
    # n_estimators=600,
    # subsample=0.8,
    # scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1]),
    use_label_encoder=False, n_jobs=-1,
    eval_metric='auc', random_state=42)
xgb_model.fit(X_train, y_train)

y_val_proba = xgb_model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))

d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [13:50:52] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.7437820223395694
              precision    recall  f1-score   support

           0       0.76      0.96      0.85     37730
           1       0.52      0.13      0.21     13211

    accuracy                           0.74     50941
   macro avg       0.64      0.54      0.53     50941
weighted avg       0.70      0.74      0.68     50941

AUC PR: 0.4471936906353448
ROC AUC: 0.739003075186744


In [ ]:
# 전체 데이터로 재학습
model_full = xgb_model
model_full.fit(X, y)

y_pred_proba = model_full.predict_proba(test_df)[:, 1]

sample_submission = pd.read_csv('Data/sample_submission.csv')
sample_submission['probability'] = y_pred_proba
sample_submission.to_csv('./base_line.csv', index=False)

## 랜덤 서치

In [10]:
# 전처리 마친 데이터 불러오기
train_df = pd.read_csv("train_e.csv")
test_df = pd.read_csv("test_e.csv")
X = train_df.drop(["임신 성공 여부"], axis=1)
y = train_df["임신 성공 여부"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
xgb_model = XGBClassifier(
    use_label_encoder=False, n_jobs=-1,
    eval_metric='auc', random_state=42)

param_dist = {
    'n_estimators': [100, 300, 500, 700],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6]
}
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=50, 
    scoring='roc_auc', 
    cv=5,  
    verbose=2,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)
print("Best Parameters:", random_search.best_params_)

best_xgb = random_search.best_estimator_
y_val_proba = best_xgb.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))
print("AUC PR:", average_precision_score(y_val, y_val_proba))
print("ROC AUC:", roc_auc_score(y_val, y_val_proba))

Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\env\envs\aimers\Lib\site-packages\xgboost\core.py:158: UserWarning: [14:10:38] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best Parameters: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1}
Accuracy: 0.7449205944131446
              precision    recall  f1-score   support

           0       0.76      0.97      0.85     37730
           1       0.54      0.11      0.19     13211

    accuracy                           0.74     50941
   macro avg       0.65      0.54      0.52     50941
weighted avg       0.70      0.74      0.68     50941

AUC PR: 0.4524867508583737
ROC AUC: 0.7413527162337291


In [ ]:
from sklearn.linear_model import LogisticRegression